# Week 4 — Ontology

This week formalizes something Weeks 1 and 3 only handled empirically.

- **Week 1 Day 1** found that a weak embedding model ranked a "customer support ticket" (a complaint) *closer* to "customer placed an order" (an order) than to its true pair — a misranking that was fixed only by switching embedding models. Nothing *prevented* that confusion; we just got lucky with a better model.
- **Week 3** kept `Order` and `InventoryReplenishment` separate by giving them no edge in the Neo4j graph. That worked too — but only because nobody happened to draw that edge. Neo4j has no concept that these are fundamentally different *kinds* of thing; `:Order` and `:InventoryReplenishment` are just string labels to it.

This week's question: can we state, as a **formal rule a machine can check**, that two classes of thing can never overlap — so that the Week 1 mistake becomes a detectable contradiction instead of a silent ranking error?

**What this notebook builds, end to end**

| Day | What | Output |
|---|---|---|
| 1 | Declare two classes as disjoint, commit the Week 1 mistake on purpose | A reasoner that notices but doesn't stop you |
| 2 | Read the reasoner's error properly; build a fuller ontology (classes, hierarchy, properties) | `reasoner_errors()` |
| 3 | The **write-time gate**: check every proposed fact on a scratch copy before it is loaded | `check_proposal()` |
| 4 | A `KnowledgeBase` that only loads facts that pass the gate (RDF always, Neo4j if configured) | accepted / rejected log |
| 5 | **Query-time**: inferred facts plus rejected-assertion notes go into the LLM prompt | `ask()` |

**Setup** (run once, Colab or local):
```
pip install rdflib owlrl neo4j openai python-dotenv
```
Neo4j and OpenAI are **optional** here: without credentials the notebook runs fully in memory and prints the prompt instead of calling the LLM.


In [ ]:
!pip install rdflib owlrl neo4j openai python-dotenv -q

## Day 1 — Declaring formal disjointness

We build a tiny ontology with `rdflib`:
- Two classes, `Order` and `Complaint`.
- An `owl:disjointWith` axiom between them — the formal claim that **no individual can ever be both at once**.
- One real instance of each, mirroring Week 1's actual `[0]` (an order) and `[7]` (a complaint).

Run this first and just confirm it builds — you should see 7 triples (2 class declarations, 1 disjointness axiom, 2 `rdf:type` assertions, 2 labels).

In [ ]:
from rdflib import Graph, Namespace, RDF, RDFS, OWL, Literal, URIRef
from rdflib.namespace import XSD

ns = Namespace("http://example.org/biz#")
g = Graph()
g.bind("biz", ns)

# Declare two classes
g.add((ns.Order, RDF.type, OWL.Class))
g.add((ns.Complaint, RDF.type, OWL.Class))

# The key formal move: these classes can NEVER share an individual
g.add((ns.Order, OWL.disjointWith, ns.Complaint))

# Create one real instance of each — mirroring Week 1's actual [0] and [7]
g.add((ns.order_0, RDF.type, ns.Order))
g.add((ns.order_0, RDFS.label, Literal("Customer placed an order for laptops")))

g.add((ns.complaint_7, RDF.type, ns.Complaint))
g.add((ns.complaint_7, RDFS.label, Literal("Customer support ticket opened for defective item")))

print(f"Triples before reasoning: {len(g)}")
for s, p, o in g:
    print(s, p, o)

## Day 1 (continued) — deliberately committing the Week 1 mistake

Now we deliberately do what a weak embedding ranking effectively encouraged: assert that `complaint_7` is **also** an `Order`. Then we run an actual OWL reasoner (`owlrl`) over the graph and see what it does with a graph that now contains a logical contradiction.

Run this and report back exactly what you see — does it raise an error, print nothing, or do something else? That result decides what Day 2 builds on.

In [ ]:
import owlrl

# Deliberately misclassify complaint_7 as ALSO an Order —
# this is precisely the kind of error a weak embedding model's
# ranking could lead someone to encode by hand
g.add((ns.complaint_7, RDF.type, ns.Order))

print(f"Triples after the bad assertion: {len(g)}")

# Run OWL-RL inference/consistency checking over the graph
owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(g)

print(f"Triples after reasoning: {len(g)}")

**What you saw on Day 1:** no exception, and the triple count jumped from 8 to about 133. Most of that jump is boilerplate OWL-RL adds to *every* graph (built-in axioms, every class being a subclass of itself, ...). The count tells you nothing about consistency.

The signal is a small error node the reasoner writes into the graph. The next cell finds it.

In [ ]:
# Read the contradiction the reasoner recorded (Day 1 graph `g`)
DAML = Namespace("http://www.daml.org/2002/03/agents/agent-ont#")

for bn in g.subjects(RDF.type, DAML.ErrorMessage):
    for msg in g.objects(bn, DAML.error):
        print("REASONER ERROR:", msg)

## Day 2 — A fuller ontology, and one function that reads the verdict

Day 1 had two classes. The Week 3 graph also had Shipments, Customers, Warehouses and Products, and relationships between them. We now declare all of that formally:

- **Hierarchy:** `Order`, `Complaint`, `InventoryReplenishment` and `Shipment` are all kinds of `BusinessEvent`.
- **Disjointness:** the four event types never overlap with each other; `Customer`, `Warehouse` and `Product` never overlap with each other.
- **Properties with domain and range:** `placedBy` goes from an `Order` to a `Customer`, and so on. This matters: if a record uses `placedBy`, the reasoner **infers** its source is an `Order`, even if nobody labelled it that. So a mislabelled node can be caught through its *edges*, not only its label.

`reasoner_errors(graph)` runs the reasoner on a **copy** (it never alters your graph) and returns the contradictions as plain strings.

In [ ]:
from itertools import combinations
import copy

EVENT_CLASSES  = ["Order", "Complaint", "InventoryReplenishment", "Shipment"]
THING_CLASSES  = ["Customer", "Warehouse", "Product"]

# (property, domain class, range class) -- mirrors the Week 3 relationships
PROPERTIES = [
    ("placedBy",    "Order",                  "Customer"),
    ("fulfilledBy", "Order",                  "Shipment"),
    ("restocks",    "InventoryReplenishment", "Product"),
    ("atWarehouse", "InventoryReplenishment", "Warehouse"),
]

def build_ontology():
    o = Graph()
    o.bind("biz", ns)
    for cls in ["BusinessEvent"] + EVENT_CLASSES + THING_CLASSES:
        o.add((ns[cls], RDF.type, OWL.Class))
    for cls in EVENT_CLASSES:
        o.add((ns[cls], RDFS.subClassOf, ns.BusinessEvent))
    for group in (EVENT_CLASSES, THING_CLASSES):
        for a, b in combinations(group, 2):
            o.add((ns[a], OWL.disjointWith, ns[b]))
    for prop, dom, rng in PROPERTIES:
        o.add((ns[prop], RDF.type, OWL.ObjectProperty))
        o.add((ns[prop], RDFS.domain, ns[dom]))
        o.add((ns[prop], RDFS.range,  ns[rng]))
    return o


def closure_of(graph):
    """Reasoner output on a COPY -- the input graph is never modified."""
    test = Graph()
    for t in graph:
        test.add(t)
    owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(test)
    return test


def reasoner_errors(graph):
    """List of contradiction messages the reasoner found (empty = consistent)."""
    closed = closure_of(graph)
    msgs = set()
    for bn in closed.subjects(RDF.type, DAML.ErrorMessage):
        for m in closed.objects(bn, DAML.error):
            msgs.add(str(m))
    return sorted(msgs)


ontology = build_ontology()
print("Ontology triples:", len(ontology))
print("Errors in the bare ontology (should be []):", reasoner_errors(ontology))

## Day 3 — The write-time gate

The reasoner *reports*; it does not *refuse*. So the check has to sit **before** the write.

For each proposed batch of nodes and edges:
1. Translate Neo4j-style labels and relationship types into ontology terms (the **mapping layer** — Neo4j labels are just strings, the ontology doesn't know about them).
2. Add them to a **scratch copy** of what is already accepted.
3. Run the reasoner on the copy.
4. If it produces a contradiction that was **not already there**, reject the batch and say why. Otherwise accept.

Anything not in the mapping (an unknown label or relationship) is rejected too, which also keeps arbitrary strings out of any Cypher we later build.

In [ ]:
# Mapping layer: graph-database vocabulary -> ontology vocabulary
LABEL_TO_CLASS = {name: ns[name] for name in EVENT_CLASSES + THING_CLASSES}
REL_TO_PROP = {
    "PLACED_BY":    ns.placedBy,
    "FULFILLED_BY": ns.fulfilledBy,
    "RESTOCKS":     ns.restocks,
    "AT_WAREHOUSE": ns.atWarehouse,
}

def to_triples(nodes, edges):
    triples = []
    for n in nodes:
        triples.append((ns[n["id"]], RDF.type, LABEL_TO_CLASS[n["label"]]))
    for e in edges:
        triples.append((ns[e["from"]], REL_TO_PROP[e["rel"]], ns[e["to"]]))
    return triples


def check_proposal(accepted_graph, nodes, edges):
    """Returns (ok, reasons). Never modifies accepted_graph."""
    try:
        triples = to_triples(nodes, edges)
    except KeyError as missing:
        return False, [f"Not in the ontology mapping: {missing}"]

    already = set(reasoner_errors(accepted_graph))

    scratch = Graph()
    for t in accepted_graph:
        scratch.add(t)
    for t in triples:
        scratch.add(t)

    new_errors = [m.replace(str(ns), "biz:") for m in reasoner_errors(scratch) if m not in already]
    return (len(new_errors) == 0), new_errors


# Quick check using the Day 1 mistake, in the new vocabulary
ok, why = check_proposal(
    ontology,
    nodes=[{"id": "CMP-7", "label": "Complaint"}, {"id": "CMP-7", "label": "Order"}],
    edges=[],
)
print("accepted:", ok)
for w in why:
    print(" -", w)

## Day 4 — A knowledge base that only loads what passes

`KnowledgeBase.propose()` runs the gate. If the batch passes, it is added to the in-memory RDF graph and, if a Neo4j connection is configured, written there as well. If it fails, nothing is written anywhere, and the rejection is logged.

**Neo4j is optional.** Add `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` to Colab Secrets (or `.env`) to also load accepted facts into Aura. Without them everything runs in memory. Note: with Neo4j configured, this notebook clears the database first, like Week 3 does.

In [ ]:
import os, re

def get_secret(name):
    """Colab Secrets first, then a local .env file, else None."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except Exception:
        pass
    return os.getenv(name)

driver = None
try:
    from neo4j import GraphDatabase
    _uri, _user, _pw = get_secret("NEO4J_URI"), get_secret("NEO4J_USER"), get_secret("NEO4J_PASSWORD")
    if _uri and _user and _pw:
        driver = GraphDatabase.driver(_uri, auth=(_user, _pw))
        with driver.session() as s:
            s.run("RETURN 1").single()
        print("Neo4j connected -- accepted facts will also be written there")
    else:
        print("Neo4j not configured -- running in memory only")
except Exception as e:
    driver = None
    print("Neo4j unavailable (%s) -- running in memory only" % type(e).__name__)


class KnowledgeBase:
    def __init__(self, ontology_graph, driver=None):
        self.graph = Graph()
        for t in ontology_graph:
            self.graph.add(t)
        self.driver = driver
        self.nodes = {}          # id -> {"label":..., "props":...}
        self.edges = []          # (from, rel, to)
        self.rejected = []       # {"nodes":..., "edges":..., "reasons":[...]}
        if driver:
            with driver.session() as s:
                s.run("MATCH (n) DETACH DELETE n")

    def propose(self, nodes, edges=()):
        nodes, edges = list(nodes), list(edges)
        ok, reasons = check_proposal(self.graph, nodes, edges)
        if not ok:
            self.rejected.append({"nodes": nodes, "edges": edges, "reasons": reasons})
            return False, reasons
        for t in to_triples(nodes, edges):
            self.graph.add(t)
        for n in nodes:
            self.nodes.setdefault(n["id"], {"label": n["label"], "props": {}})
            self.nodes[n["id"]]["props"].update(n.get("props", {}))
        for e in edges:
            self.edges.append((e["from"], e["rel"], e["to"]))
        self._write_neo4j(nodes, edges)
        return True, []

    def _write_neo4j(self, nodes, edges):
        if not self.driver:
            return
        # Labels / relationship types are safe to splice in: check_proposal
        # already rejected anything not in LABEL_TO_CLASS / REL_TO_PROP.
        with self.driver.session() as s:
            for n in nodes:
                s.run(f"MERGE (x:{n['label']} {{id:$id}}) SET x += $props",
                      id=n["id"], props=n.get("props", {}))
            for e in edges:
                s.run(f"MATCH (a {{id:$a}}), (b {{id:$b}}) "
                      f"MERGE (a)-[:{e['rel']}]->(b)", a=e["from"], b=e["to"])


kb = KnowledgeBase(ontology, driver)
print("Knowledge base ready")

### Replaying Week 3's facts, plus four attempts

The first batches are the two manual graphs from Week 3 notebook 3. Then four more proposals:

| # | Proposal | What should happen, and why |
|---|---|---|
| A | A new complaint `CMP-7` | accepted, nothing conflicts |
| B | Re-label `ORD-101` as a `Complaint` | rejected: `Order` and `Complaint` are disjoint |
| C | `REPL-77 PLACED_BY CUST-1` | rejected **through the edge**: `placedBy` has domain `Order`, so the reasoner infers `REPL-77` is an `Order`, which clashes with `InventoryReplenishment` |
| D | `ORD-101 PLACED_BY WH-1` | rejected: `placedBy` has range `Customer`, so the warehouse would be inferred to be a customer |

Case C is the Week 3 lesson made formal: `Order` and `InventoryReplenishment` stayed apart only because nobody drew that edge. Now drawing it is refused.

In [ ]:
def report(title, result):
    ok, reasons = result
    print(("ACCEPTED  " if ok else "REJECTED  ") + title)
    for r in reasons:
        print("     reason:", r)

# --- Week 3 graph 1: customer -> order -> shipment
report("Customer, Order, Shipment (graph 1)", kb.propose(
    nodes=[
        {"id": "CUST-1", "label": "Customer",  "props": {"name": "Acme Corp"}},
        {"id": "ORD-101","label": "Order",     "props": {"product": "Printer cartridges", "status": "Fulfilled"}},
        {"id": "SHP-501","label": "Shipment"},
    ],
    edges=[
        {"from": "ORD-101", "rel": "PLACED_BY",    "to": "CUST-1"},
        {"from": "ORD-101", "rel": "FULFILLED_BY", "to": "SHP-501"},
    ]))

# --- Week 3 graph 2: replenishment -> product / warehouse
report("Replenishment, Product, Warehouse (graph 2)", kb.propose(
    nodes=[
        {"id": "WH-1",   "label": "Warehouse", "props": {"name": "Bengaluru DC"}},
        {"id": "PRD-1",  "label": "Product",   "props": {"name": "Printer cartridges"}},
        {"id": "REPL-77","label": "InventoryReplenishment"},
    ],
    edges=[
        {"from": "REPL-77", "rel": "RESTOCKS",     "to": "PRD-1"},
        {"from": "REPL-77", "rel": "AT_WAREHOUSE", "to": "WH-1"},
    ]))

print()
report("A: new complaint CMP-7", kb.propose(
    nodes=[{"id": "CMP-7", "label": "Complaint", "props": {"text": "Support ticket for defective item"}}]))

report("B: re-label ORD-101 as a Complaint", kb.propose(
    nodes=[{"id": "ORD-101", "label": "Complaint"}]))

report("C: REPL-77 PLACED_BY CUST-1", kb.propose(
    nodes=[],
    edges=[{"from": "REPL-77", "rel": "PLACED_BY", "to": "CUST-1"}]))

report("D: ORD-101 PLACED_BY WH-1", kb.propose(
    nodes=[],
    edges=[{"from": "ORD-101", "rel": "PLACED_BY", "to": "WH-1"}]))

print()
print("Accepted nodes:", len(kb.nodes), "| edges:", len(kb.edges), "| rejected batches:", len(kb.rejected))

## Day 5 — Query time: useful inferred facts, plus a note about what was refused

The gate keeps contradictions out, so the prompt does not need to warn the model about bad data. What *is* worth putting in the prompt:

1. **Inferred types**: `ORD-101` was only ever labelled `Order`, but the ontology also tells us it is a `BusinessEvent`.
2. **Direct relationships** recorded for the entities in the question.
3. **Rejected assertions** about those entities. If someone asks "is REPL-77 an order?", the honest answer includes that an attempt to record it as one was refused, and why.

Entity lookup here is a simple ID match (`ORD-101`, `REPL-77`...). In the full pipeline, the Week 2 `retrieve()` step would find the entities, and this function would enrich them. Facts go in as plain lines with record IDs next to every claim, the same lesson as Day 2 of the RAG project. The call uses `temperature=0`.

In [ ]:
ID_PATTERN = re.compile(r"\b[A-Z]{2,5}-\d{1,4}\b")

def inferred_types(kb, entity_id):
    closed = closure_of(kb.graph)
    names = set()
    for o in closed.objects(ns[entity_id], RDF.type):
        if str(o).startswith(str(ns)):
            names.add(str(o)[len(str(ns)):])
    return sorted(names)


def facts_for(kb, question):
    ids = [i for i in ID_PATTERN.findall(question) if i in kb.nodes]
    lines = []
    for i in ids:
        declared = kb.nodes[i]["label"]
        types = inferred_types(kb, i)
        lines.append(f"Fact: {i} is a {declared}; the ontology also classes it as: {', '.join(types)}.")
        for (a, rel, b) in kb.edges:
            if a == i:
                lines.append(f"Fact: {a} --{rel}--> {b}")
        for rej in kb.rejected:
            touched = {n["id"] for n in rej["nodes"]} | {e["from"] for e in rej["edges"]} | {e["to"] for e in rej["edges"]}
            if i in touched:
                lines.append(f"Fact: an attempt to record a change involving {i} was REFUSED by the ontology check: {'; '.join(rej['reasons'])}")
    return ids, lines


def build_prompt(kb, question):
    ids, lines = facts_for(kb, question)
    facts = "\n".join(lines) if lines else "(no matching entities found)"
    return f"""Answer using ONLY the facts below. Treat the question's wording as a claim to check, not as evidence.
If the facts don't settle it, say so. Quote record IDs from the facts.

Facts:
{facts}

Question: {question}

Answer:"""


try:
    from openai import OpenAI
    _key = get_secret("OPENAI_API_KEY")
    client = OpenAI(api_key=_key) if _key else None
except Exception:
    client = None
print("LLM client ready" if client else "No OPENAI_API_KEY -- ask() will print the prompt instead of calling the LLM")


def ask(kb, question):
    prompt = build_prompt(kb, question)
    if client is None:
        print(prompt)
        return prompt
    resp = client.chat.completions.create(
        model="gpt-4o-mini", temperature=0,
        messages=[{"role": "user", "content": prompt}])
    answer = resp.choices[0].message.content
    print("Q:", question)
    print("A:", answer)
    return answer


for q in ["Is REPL-77 an order?",
          "Did Acme Corp's order ORD-101 get re-labelled as a complaint?",
          "What kind of record is CMP-7?"]:
    ask(kb, q)
    print("-" * 70)

## Wrap-up and exercises

**What was built:** a pipeline where a formal rule ("these classes never overlap", "this relationship goes from an Order to a Customer") is checked on every write, so a Week 1-style confusion is refused at the door instead of ranked silently.

**Things to try, in order**
1. Add a new disjoint pair of your own (for example `Complaint` vs `Shipment` is already covered; try adding a `Refund` class that is disjoint from `Order`) and write a proposal that violates it.
2. Propose a batch with two facts where only the *second* is bad. Is the whole batch refused? Is that what you want?
3. Declare a rule you *forgot* to make disjoint and confirm the gate lets a bad fact through. What does this say about who maintains the ontology?
4. Time `check_proposal` on a graph with a few thousand triples. When would you move from one-reasoner-run-per-write to batching?
5. Wire `facts_for()` into the Day 2 RAG notebook so it enriches the entities found by `retrieve()`.

**Known limits**
- The reasoner enforces only what the ontology states; it does not guess unstated rules.
- One reasoner run per write is fine for small graphs; larger ones need batching or incremental checking.
- The ID lookup in `facts_for()` is deliberately simple.